# Four-model ink ensemble (PHerc0343 "First Letters" models) on our test surfaces

Models (Hugging Face, sha256-checked against `Nieuwlaar/pherc343-first-letters/ink/models.tsv`, in `models/villa_ensemble/`):
`dnative` (Nieuwlaar/ink9um-dense-native), `v8in-1447` (YoussefMoNader/ink-8um-v8in-pherc1447-loo-w062, replaces ink_9um seed 42), `ink9-43` (scrollprize/ink_9um hybrid_3d2d seed 43), `hecate` (scrollprize/hecate 9.6µm).

Inference runs in the isolated villa env (`/opt/villa-venv`, villa @ `0fb38c45`) via `utils/villa_driver.py`, exactly as the reference `run_ink.sh`.
Inputs: our 28-layer renders copied to uint8 zarr groups (`ves_zarrs_villa/<id>.zarr/0`). Ensemble = reference `ensemble_maps.py` (mean of per-model z-scores).

Per fragment, saved (not displayed) to `output/test_visualizations/villa_ensemble/`:
- `<name>_4models_d+0.png` — 2x2 grid of the four models, every panel at the render's native resolution
- `<name>_ensemble_d+0.png` — the ensemble at native resolution

Run the sanity check (seg46527, PHerc0814) first: if ink is not visible, switch `DIRECTION` in cell 2.

In [ ]:
import os
import gc
import subprocess
import sys
import time

import numpy as np

if sys.executable.startswith("/opt/villa-venv"):
    raise RuntimeError("select the system Python 3.10 kernel (/usr/bin/python3); /opt/villa-venv is only used "
                       "internally for model inference")

REPO = "/vesuvius"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import importlib
from utils import villa_ensemble as ve
ve = importlib.reload(ve)

ZARR_DIR = os.getenv("VESUVIUS_ZARR_PATH", os.path.join(REPO, "ves_zarrs2"))
OUT_DIR = os.path.join(REPO, "output", "test_visualizations", "villa_ensemble")
# depths d computed (stretch statistics pool over all of them); 28-layer renders allow d in [-5, 5]
DEPTHS = (0,)
WRITE_DEPTH = 0
# layer order that reads the text side of our (training-oriented) renders: seg46527 ink 0.67 vs bg 0.14 forward, none reverse
DIRECTION = "forward"
COMPILE = False
TITLES = {"dnative": "dense_native (16k)", "hecate": "Hecate 9.6µm",
          "v8in-1447": "v8-in PHerc1447 LOO-w062", "ink9-42": "ink_9um seed 42", "ink9-43": "ink_9um seed 43"}

FRAGMENTS = {
    "auto_grown_20260814140748": 20260814140748,
    "auto_grown_20260717193517": 20260717193517,
    "auto_grown_20260720090842": 20260720090842,
    "auto_grown_20260723112922652_merged": 20260723112922,
    "auto_grown_20260921094413486": 20260921094413,
    "auto_grown_20260918132724424": 20260918132724,
    "auto_grown_20260922073234974": 20260922073234,
    "auto_grown_20260922161631422_merged": 20260922161631,
    "PHerc0800_z9264_w040_abf": 20260928000001,
    "PHerc0125_z15344_w040_abf": 20260928000002,
    "PHerc0211_z7312_w080_abf": 20260928000003,
    "PHerc0211_z5520_w040_abf": 20260928000004,
    "PHerc0813_z13088_w040_abf": 20260928000007,
    # sanity check: training fragment seg46527 (PHerc0814), native surface volume with known ink
    "seg46527_PHerc0814": 20260226000000,
}


def ensure_assembled(scroll_id):
    """render a test zarr from its tifxyz mesh when absent; the assembler writes the mask after the last layer"""
    zarr_path = os.path.join(ZARR_DIR, f"{scroll_id}.zarr")
    if scroll_id == "20260226000000":  # training fragment, assembled by assemble_training_segments.py
        return
    mask_path = os.path.join(REPO, "masks", f"{scroll_id}.png")
    if not os.path.isdir(zarr_path):
        subprocess.run([sys.executable, os.path.join(REPO, "assemble_test_segments.py"), "--only", str(scroll_id),
                        "--out-dir", ZARR_DIR], cwd=REPO, check=True)
    zarray = os.path.join(zarr_path, ".zarray")
    if not (os.path.isfile(mask_path) and os.path.getmtime(mask_path) >= os.path.getmtime(zarray)):
        raise RuntimeError(f"{zarr_path} is still rendering (or was interrupted: rerun the assembler with --force --only {scroll_id})")


def render(name):
    scroll_id = str(FRAGMENTS[name])
    t0 = time.time()
    ensure_assembled(scroll_id)
    ve.ensure_villa_input(scroll_id, ZARR_DIR)
    ve.run_models(scroll_id, DEPTHS, [DIRECTION], compile_model=COMPILE)
    print("[ensemble] stretching + z-score ensemble ...", flush=True)
    stretched, ensemble, stats = ve.ensemble(scroll_id, DEPTHS, DIRECTION, WRITE_DEPTH)
    tag = f"d{WRITE_DEPTH:+d}"
    grid_path = os.path.join(OUT_DIR, f"{name}_4models_{tag}.png")
    ens_path = os.path.join(OUT_DIR, f"{name}_ensemble_{tag}.png")
    ve.save_grid([(f"{TITLES[k]}  {tag} {DIRECTION}", stretched[k]) for k, _ in ve.MODELS], 2, grid_path)
    ve.save_ensemble_figure(scroll_id, ensemble, f"4-model ensemble (mean z-score)  {tag} {DIRECTION}", ens_path)
    del stretched, ensemble, stats
    gc.collect()
    print(f"[saved] {grid_path}\n[saved] {ens_path}\n[done] {name} in {time.time() - t0:.0f}s")


def render_depths(name, depths=tuple(range(-5, 6))):
    """depth sweep into OUT_DIR/<name>_depths/; stats pool over all depths, models skip depths they cannot reach"""
    scroll_id = str(FRAGMENTS[name])
    t0 = time.time()
    ensure_assembled(scroll_id)
    ve.ensure_villa_input(scroll_id, ZARR_DIR)
    ve.run_models(scroll_id, depths, [DIRECTION], compile_model=COMPILE)
    out_dir = os.path.join(OUT_DIR, f"{name}_depths")
    for d in depths:
        stretched, ensemble, stats = ve.ensemble(scroll_id, depths, DIRECTION, d)
        tag = f"d{d:+d}"
        used = [k for k, _ in ve.MODELS if not np.isnan(stretched[k]).all()]
        titles = [f"{TITLES[k]}  {tag} {DIRECTION}" + ("" if k in used else "  (no map: out of depth range)")
                  for k, _ in ve.MODELS]
        ve.save_grid([(title, stretched[k]) for title, (k, _) in zip(titles, ve.MODELS)], 2,
                     os.path.join(out_dir, f"{name}_4models_{tag}.png"))
        ve.save_ensemble_figure(scroll_id, ensemble, f"{len(used)}-model ensemble (mean z-score)  {tag} {DIRECTION}",
                                os.path.join(out_dir, f"{name}_ensemble_{tag}.png"))
        del stretched, ensemble, stats
        gc.collect()
        print(f"[saved] {tag}: {len(used)} models", flush=True)
    print(f"[done] {name} depths {depths[0]:+d}..{depths[-1]:+d} in {time.time() - t0:.0f}s -> {out_dir}")


def render_all(names=None):
    """overnight: render every fragment, logging failures instead of stopping; finished maps are cached"""
    failed = []
    for name in names or FRAGMENTS:
        try:
            render(name)
        except Exception as exc:
            failed.append(name)
            print(f"[FAILED] {name}: {exc}", flush=True)
    print(f"[render_all] {len(failed)} failed: {failed}")


print(f"villa python: {ve.VILLA_PY} | checkpoints: {ve.CKPT_DIR} | out: {OUT_DIR} | depths={DEPTHS} direction={DIRECTION}")


villa python: /opt/villa-venv/bin/python | checkpoints: /vesuvius/models/villa_ensemble | out: /vesuvius/output/test_visualizations/villa_ensemble | depths=(0,) direction=forward


In [2]:
# --- sanity check: seg46527 (PHerc0814, 9.362µm), known ink ---
render("seg46527_PHerc0814")

[villa] 20260226000000: all 4 maps cached
[ensemble] stretching + z-score ensemble ...
[saved] /vesuvius/output/test_visualizations/villa_ensemble/seg46527_PHerc0814_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/seg46527_PHerc0814_ensemble_d+0.png
[done] seg46527_PHerc0814 in 10s


In [3]:
# --- test fragment 1: PHerc0813 ---
render("auto_grown_20260814140748")

[villa] 20260814140748: all 4 maps cached
[ensemble] stretching + z-score ensemble ...
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260814140748_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260814140748_ensemble_d+0.png
[done] auto_grown_20260814140748 in 27s


In [4]:
# --- test fragment 2: PHerc0211 merged ---
render("auto_grown_20260717193517")

20260717193517: 100%|██████████| 2.00/2 maps [31:02<00:00] , hybrid_3d2d-seed43__L28s6.tif 7284/7284 patches

[villa] 20260717193517: 2 maps in 1862s -> /vesuvius/output/villa_ensemble_maps/20260717193517
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260717193517_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260717193517_ensemble_d+0.png
[done] auto_grown_20260717193517 in 1917s


In [5]:
# --- test fragment 3: PHerc1203 ---
render("auto_grown_20260720090842")

20260720090842: 100%|██████████| 4.00/4 maps [10:01<00:00] , hybrid_3d2d-seed43__L28s6.tif 1881/1881 patches

[villa] 20260720090842: 4 maps in 601s -> /vesuvius/output/villa_ensemble_maps/20260720090842
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260720090842_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260720090842_ensemble_d+0.png
[done] auto_grown_20260720090842 in 617s


In [6]:
# --- test fragment 5: PHerc0826 merged ---
render("auto_grown_20260723112922652_merged")

20260723112922: 100%|██████████| 4.00/4 maps [11:35<00:00] , hybrid_3d2d-seed43__L28s6.tif 2244/2244 patches

[villa] 20260723112922: 4 maps in 695s -> /vesuvius/output/villa_ensemble_maps/20260723112922
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260723112922652_merged_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260723112922652_merged_ensemble_d+0.png
[done] auto_grown_20260723112922652_merged in 716s


In [7]:
# --- test fragment 6: PHerc0846A ---
render("auto_grown_20260921094413486")

20260921094413: 100%|██████████| 4.00/4 maps [31:41<00:00] , hybrid_3d2d-seed43__L28s6.tif 5725/5725 patches

[villa] 20260921094413: 4 maps in 1901s -> /vesuvius/output/villa_ensemble_maps/20260921094413
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260921094413486_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260921094413486_ensemble_d+0.png
[done] auto_grown_20260921094413486 in 1954s


In [8]:
# --- test fragment 7: PHerc0175A (8.64µm) ---
render("auto_grown_20260918132724424")

20260918132724: 100%|██████████| 4.00/4 maps [15:57<00:00] , hybrid_3d2d-seed43__L28s6.tif 3070/3070 patches

[villa] 20260918132724: 4 maps in 958s -> /vesuvius/output/villa_ensemble_maps/20260918132724
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260918132724424_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260918132724424_ensemble_d+0.png
[done] auto_grown_20260918132724424 in 985s


In [9]:
# --- test fragment 8: PHerc0306B (8.64µm) ---
render("auto_grown_20260922073234974")

20260922073234: 100%|██████████| 4.00/4 maps [24:01<00:00] , hybrid_3d2d-seed43__L28s6.tif 4452/4452 patches

[villa] 20260922073234: 4 maps in 1441s -> /vesuvius/output/villa_ensemble_maps/20260922073234
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260922073234974_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260922073234974_ensemble_d+0.png
[done] auto_grown_20260922073234974 in 1483s


In [10]:
# --- test fragment 9: PHerc0800 merged (8.64µm) ---
render("auto_grown_20260922161631422_merged")

20260922161631: 100%|██████████| 4.00/4 maps [39:44<00:00] , hybrid_3d2d-seed43__L28s6.tif 7282/7282 patches

[villa] 20260922161631: 4 maps in 2384s -> /vesuvius/output/villa_ensemble_maps/20260922161631
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260922161631422_merged_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/auto_grown_20260922161631422_merged_ensemble_d+0.png
[done] auto_grown_20260922161631422_merged in 2459s


In [11]:
# --- test fragment 11: PHerc0800 expanded auto-winder segment (8.64µm) ---
render("PHerc0800_z9264_w040_abf")

20260928000001: 100%|██████████| 4.00/4 maps [10:38<00:00] , hybrid_3d2d-seed43__L28s6.tif 1982/1982 patches

[villa] 20260928000001: 4 maps in 638s -> /vesuvius/output/villa_ensemble_maps/20260928000001
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0800_z9264_w040_abf_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0800_z9264_w040_abf_ensemble_d+0.png
[done] PHerc0800_z9264_w040_abf in 654s


In [12]:
# --- test fragment 12: PHerc0125 expanded auto-winder segment ---
render("PHerc0125_z15344_w040_abf")

20260928000002: 100%|██████████| 4.00/4 maps [04:13<00:00] , hybrid_3d2d-seed43__L28s6.tif 824/824 patches

[villa] 20260928000002: 4 maps in 254s -> /vesuvius/output/villa_ensemble_maps/20260928000002
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0125_z15344_w040_abf_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0125_z15344_w040_abf_ensemble_d+0.png
[done] PHerc0125_z15344_w040_abf in 260s


In [13]:
# --- test fragment 13: PHerc0211 expanded auto-winder segment ---
render("PHerc0211_z7312_w080_abf")

20260928000003: 100%|██████████| 4.00/4 maps [38:33<00:00] , hybrid_3d2d-seed43__L28s6.tif 6918/6918 patches

[villa] 20260928000003: 4 maps in 2313s -> /vesuvius/output/villa_ensemble_maps/20260928000003
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0211_z7312_w080_abf_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0211_z7312_w080_abf_ensemble_d+0.png
[done] PHerc0211_z7312_w080_abf in 2389s


In [14]:
# --- PHerc0211 auto-winder segment z5520 ---
render("PHerc0211_z5520_w040_abf")

20260928000004: 100%|██████████| 4.00/4 maps [21:49<00:00] , hybrid_3d2d-seed43__L28s6.tif 3901/3901 patches

[villa] 20260928000004: 4 maps in 1309s -> /vesuvius/output/villa_ensemble_maps/20260928000004
[ensemble] stretching + z-score ensemble ...


[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0211_z5520_w040_abf_4models_d+0.png
[saved] /vesuvius/output/test_visualizations/villa_ensemble/PHerc0211_z5520_w040_abf_ensemble_d+0.png
[done] PHerc0211_z5520_w040_abf in 1340s


In [ ]:
# --- PHerc0813 auto-winder segment z13088 (v8-in PHerc1447 fine-tune; applies to later cells too) ---
ve.set_slot3("v8in-1447")
render("PHerc0813_z13088_w040_abf")


In [ ]:
# --- PHerc0813 z13088 depth sweep d = -5..+5 (v8-in only reaches -2..+2 on 28-layer renders) ---
ve.set_slot3("v8in-1447")
render_depths("PHerc0813_z13088_w040_abf")
